# Lecture 11 - Quiz, and Least Squares Regression

We will start today's lecture with a quiz. The quiz will start promptly at 9:05 AM, and last 10 minutes. Please bring a pencil or pen.

Last lecture, we discussed some techniques one can use to solve a multi-dimensional equation with multiple variables:
$$
{\bf f}({\bf x}) = 0.
$$
This lecture uses these techniques in one of the most applied examples in astrophysics: fitting a model to a dataset. For this specific lecture, we will discuss least squares regression. In the following lecture, we will introduce more general methods to fit models to multidimensional data.

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
**insert link** \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

This lecture made use of material from the following sources:\
https://zingale.github.io/computational_astrophysics/fitting/least_squares.html \
https://zingale.github.io/computational_astrophysics/fitting/fitting-to-a-line.html \
https://zingale.github.io/computational_astrophysics/fitting/generalized-linear-least-squares.html \
https://zingale.github.io/computational_astrophysics/fitting/generalized-linear-least-squares-tests.html \
https://zingale.github.io/computational_astrophysics/fitting/fitting-nonlinear.html \
https://zingale.github.io/computational_astrophysics/fitting/fitting-scipy.html \
https://zingale.github.io/computational_astrophysics/fitting/application-snia-h0.html

In [ ]:
# Nice things to have for this lecture

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sympy
from scipy import optimize

# Least Squares Regression

Imagine a dataset where we have a set of points, $(x_i, y_i)$ with
associated errors in $y$, $\sigma_i$, and we want to fit a line or
curve to them.

Our data looks like:
![fitting_illustration](https://zingale.github.io/computational_astrophysics/_images/fitting-illustration.png)

We want to create a model, $Y(x; {a_j})$ that best fits these points.
Here the values ${a_j}$ are free parameters that we will tune to get a
good fit.

For each point, we will measure the _vertical_ distance to the model:

$$\Delta_i = Y(x_i; {a_j}) - y_i$$

and we want to minimize the distance.  We do this by defining

$$\chi^2({a_j}) \equiv \sum_i \left(\frac{\Delta_i}{\sigma_i}\right )^2$$

Here, each point's distance, $\Delta_i$ is weighted by its error,
$\sigma_i$ &mdash; this ensures that points that we are least certain
of has less influence in the fit.  Minimizing $\chi^2$ with respect to
the fit parameters, ${a_j}$ is called _least squares minimization_.

The minimization procedure involves setting the derivatives of $\chi^2$ with respect to each of the ${a_j}$
to zero and solving the resulting system.

For _linear least squares_, all of the ${a_j}$ enter into $Y(x; {a_j})$ linearly, e.g., as:

$$Y(x; {a_j}) = a_1 + a_2 x + a_3 x^2$$

And the minimization process results in a linear system that can be
solved using the techniques we learned when we discussed linear
algebra.

**Note:** Even though this polynomial $Y$ here is nonlinear in $x$, it is linear in $a_j$, which means that this is still a case of _linear least squares_.

The special case of fitting a line:

$$Y(x; {a_j}) = a_1 + a_2 x$$

is called _linear regression_.

For _nonlinear least squares_, the parameters can enter in a nonlinear
fashion, and the solution methodology is considerably more complex.

# General Linear Least Squares

The _linear_ in linear least squares refers to how the parameters
appear in the fitting function, $Y$.  So something of the form:

$$Y(x; \{a_j\}) = \sum_{j=1}^M a_j \varphi_j(x)$$

is still linear in the $\{a_j\}$, even if the _basis functions_
$\{\varphi_j\}$ are nonlinear.


**Example:**
The fitting function:

$$Y(x; \{a_j\}) = a_1 + a_2 x + a_3 x^2$$

is linear in the fit parameters $\{ a_j\}$,  The basis functions in this case are:

\begin{equation}
\varphi_1 = 1, \hspace{5mm}
\varphi_2 = x, \hspace{5mm}
\varphi_3 = x^2
\end{equation}

We can apply the same technique we just did for fitting to a line for this general case.

Our $\chi^2$ is:

\begin{align*}
\chi^2(\{a_j\}) &= \sum_{i=1}^N \frac{(Y(x_i; \{a_j\}) - y_i)^2}{\sigma_i^2} \\
    &= \sum_{i=1}^N \frac{1}{\sigma_i^2} \left [
          \left (\sum_{j=1}^M a_j \varphi_j(x_i)\right ) - y_i \right ]^2
\end{align*}

We can differentiate it with respect to one of the parameters, $a_k$:

\begin{align*}
\frac{\partial \chi^2}{\partial a_k}
    &= \frac{\partial}{\partial a_k}
          \sum_{i=1}^N \frac{1}{\sigma_i^2} \left [\left (\sum_{j=1}^M a_j \varphi_j(x_i)\right ) - y_i \right ]^2 \\
    &= \sum_{i=1}^N \frac{1}{\sigma_i^2}
          \frac{\partial}{\partial a_k} \left [\left (\sum_{j=1}^M a_j \varphi_j(x_i)\right ) - y_i \right ]^2 \\
    &= 2 \sum_{i=1}^N \frac{1}{\sigma_i^2} \left [\left (\sum_{j=1}^M a_j \varphi_j(x_i)\right ) - y_i \right ] \varphi_k(x_i) = 0
\end{align*}

or rewriting:

$$\sum_{i=1}^N \sum_{j=1}^M a_j \frac{\varphi_j(x_i) \varphi_k(x_i)}{\sigma_i^2} =
   \sum_{i=1}^N \frac{y_i \varphi_k(x_i)}{\sigma_i^2}$$

We define the $N\times M$ [_design matrix_](https://en.wikipedia.org/wiki/Design_matrix) as

$$A_{ij} = \frac{\varphi_j(x_i)}{\sigma_i}$$

and the source as:

$$b_i = \frac{y_i}{\sigma_i}$$

our system is:

$$\sum_{i=1}^N \sum_{j=1}^M A_{ik} A_{ij} a_j = \sum_{i=1}^N A_{ik} b_i$$

which, by looking at which indices contract, gives us row $k$ of the linear system:

$${\bf A}^\intercal \cdot {\bf A} \cdot {\bf a} = {\bf A}^\intercal \cdot {\bf b}$$

where ${\bf A}^\intercal {\bf A}$ is an $M\times M$ matrix.

The procedure we described above is sometimes called [_ordinary least
squares_](https://en.wikipedia.org/wiki/Ordinary_least_squares).


## Linear fit revisited

For a linear fit,

$$Y(x) = a_1 + a_2 x$$

and our basis functions are: $\phi_1 = 1$ and $\phi_2 = x$.

### Design matrix and source

Our design matrix in this case is:

$${\bf A} = \left ( \begin{array}{cc}
                1/\sigma_1 & x_1 / \sigma_1 \\
                1/\sigma_2 & x_2 / \sigma_2 \\
                \vdots & \vdots \\
                1/\sigma_N & x_N / \sigma_N \\
               \end{array}\right )$$

and the source is:

$${\bf b} = \left (\begin{array}{c} y_1 / \sigma_1 \\
                                    y_2 / \sigma_2 \\
                                    \vdots \\
                                    y_N / \sigma_N \end{array} \right )$$


### Linear system

We can now use this design matrix and source to find the underlying linear system.

${\bf A}^\intercal {\bf A}$ is:

\begin{align*}
{\bf A}^\intercal{\bf A} &= \left ( \begin{array}{cccc}
                            1/\sigma_1 & 1/\sigma_2 & \cdots & 1/\sigma_N \\
                            x_1/\sigma_1 & x_2/\sigma_2 & \cdots & x_N/\sigma_N \end{array} \right )
                            \left ( \begin{array}{cc}
                1/\sigma_1 & x_1 / \sigma_1 \\
                1/\sigma_2 & x_2 / \sigma_2 \\
                \vdots & \vdots \\
                1/\sigma_N & x_N / \sigma_N \\
               \end{array}\right ) \\
               &= \left ( \begin{array}{cc} \sum_i 1/\sigma_i^2 & \sum_i x_i / \sigma_i^2 \\
                                           \sum_i x_i/\sigma_i^2 & \sum_i x_i^2 / \sigma_i^2 \end{array} \right )
\end{align*}

and ${\bf A}^\intercal {\bf A} {\bf a}$ is:

\begin{align*}
{\bf A}^\intercal {\bf A} {\bf a} &=
   \left ( \begin{array}{cc} \sum_i 1/\sigma_i^2 & \sum_i x_i / \sigma_i^2 \\
                             \sum_i x_i/\sigma_i^2 & \sum_i x_i^2 / \sigma_i^2 \end{array} \right )
   \left ( \begin{array}{c} a_1 \\ a_2 \end{array} \right ) \\
   &= \left ( \begin{array}{c} a_1 \sum_i 1/\sigma_i^2 + a_2 \sum_i x_i/\sigma_i^2 \\
                               a_1 \sum_i x_i/\sigma_i^2 + a_2 \sum_i x_i^2 /\sigma_i^2 \end{array} \right )
                               \end{align*}


${\bf A}^\intercal {\bf b}$ is:

\begin{align*}
{\bf A}^\intercal {\bf b} &= \left ( \begin{array}{cccc}
                            1/\sigma_1 & 1/\sigma_2 & \cdots & 1/\sigma_N \\
                            x_1/\sigma_1 & x_2/\sigma_2 & \cdots & x_N/\sigma_N \end{array} \right )
           \left ( \begin{array}{c}
                y_1 / \sigma_1 \\
                y_2 / \sigma_2 \\
                \vdots \\
                y_N / \sigma_N \\
               \end{array}\right ) \\
               &= \left ( \begin{array}{c} \sum_i y_i / \sigma_i^2 \\
                                           \sum_i x_i y_i / \sigma_i^2 \end{array} \right )
\end{align*}

Putting these together, this gives us 2 equations with 2 unknowns ($a_1$, $a_2$):

\begin{align*}
a_1 \sum_i \frac{1}{\sigma_i^2} + a_2 \sum_i \frac{x_i}{\sigma_i^2} &= \sum_i \frac{y_i}{\sigma_i^2} \\
a_1 \sum_i \frac{x_i}{\sigma_i^2} + a_2 \sum_i \frac{x_i^2}{\sigma_i^2} &= \sum_i \frac{x_i y_i}{\sigma_i^2}
\end{align*}

This is precisely the system we saw before.

# Fitting to a Line

Let's consider the case of fitting data to a line.  Our model has the form:

$$Y(x; a_1, a_2) = a_1 + a_2 x$$

and our fit appears as:

$$\chi^2(a_1, a_2) = \sum_{i=1}^N \frac{(a_1 + a_2 x_i - y_i)^2}{\sigma_i^2}$$

We want to minimize $\chi^2(a_1, a_2)$.

We will actually care about the [_reduced chi-square_](https://en.wikipedia.org/wiki/Reduced_chi-squared_statistic), which is scaled by the number of degrees of freedom, $N - M$, where $N$ is the number of data points and $M$ is the number of fitting parameters.

Generally we want the $\chi^2 < 1$ for a fit to be considered "good".

We start by differentiating with respect to the fit parameters and setting the derivatives to zero:

\begin{align*}
\frac{\partial \chi^2}{\partial a_1} &= 
  2 \sum_{i=1}^{N} \frac{a_1 + a_2 x_i - y_i}{\sigma_i^2} = 0 \\
\frac{\partial \chi^2}{\partial a_2} &= 
  2 \sum_{i=1}^{N} \frac{a_1 + a_2 x_i - y_i}{\sigma_i^2}  x_i= 0
\end{align*}

Separating the terms, we have:

\begin{align*}
a_1 \sum_{i=1}^N \frac{1}{\sigma_i^2} + a_2 \sum_{i=1}^N \frac{x_i}{\sigma_i^2} - \sum_{i=1}^N \frac{y_i}{\sigma_i^2} &= 0 \\
a_1 \sum_{i=1}^N \frac{x_i}{\sigma_i^2} + a_2 \sum_{i=1}^N \frac{x_i^2}{\sigma_i^2} - \sum_{i=1}^N \frac{x_i y_i}{\sigma_i^2} &= 0
\end{align*}

This is a linear system with 2 equations and 2 unknowns.

Let's define:

\begin{align*}
C &= \sum_{i=1}^N \frac{1}{\sigma_i^2} \\
S_x &= \sum_{i=1}^N \frac{x_i}{\sigma_i^2} \\
S_y &= \sum_{i=1}^N \frac{y_i}{\sigma_i^2} \\
S_{x^2} &= \sum_{i=1}^N \frac{x_i^2}{\sigma_i^2} \\
S_{xy} &= \sum_{i=1}^N \frac{x_i y_i}{\sigma_i^2}
\end{align*}

Then our system is:

\begin{align*}
a_1 C + a_2 S_x - S_y &= 0 \\
a_1 S_x + a_2 S_{x^2} - S_{xy} &= 0
\end{align*}

We can solve this easily:

$$a_1 = \frac{S_{x^2} S_y - S_x S_{xy}}{C S_{x^2} - S_x^2}$$
$$a_2 = \frac{C S_{xy} - S_x S_y}{C S_{x^2} - S_x^2}$$

## Example data

Let's make some sample data that we perturb with Gaussian-normalized noise.  We'll use the NumPy [standard_normal](https://numpy.org/doc/stable/reference/random/generated/numpy.random.standard_normal.html) function.

Let's first see how this works.  Let's take a large number of samples and also plot a Gaussian distribution:

$$y(x) = \frac{1}{\sigma \sqrt{2\pi}}  e^{-x^2/(2 \sigma^2)}$$

In [ ]:
N = 10000

rng = np.random.default_rng()
r = rng.standard_normal(N)

fig, ax = plt.subplots()
ax.hist(r, density=True, bins=20)

x = np.linspace(-5, 5, 200)
sigma = 1.0
ax.plot(x, np.exp(-x**2/(2*sigma**2)) / (sigma*np.sqrt(2.0*np.pi)), lw=2)
ax.set_xlabel("x")

plt.show()

Now we can make some _experimental data_.  This will be data that follows a line, but is perturbed by a Gaussian-normalized random number, to give it some experimental error.

In [ ]:
def y_experiment(a1, a2, sigma, x):
    """ return the experimental data and error in a linear + random
    fashion; a1 is the intercept, a2 is the slope, and sigma is the
    error scale"""

    N = len(x)

    rng = np.random.default_rng()
    r = sigma * rng.standard_normal(N)

    yerr = sigma * np.ones_like(rng)
    return a1 + a2*x + r, yerr

Now we can make the data that we want to fit to.

In [ ]:
# number of data points
N = 40

# one-sigma error
sigma = 25.0

# independent variable
x = np.linspace(0.0, 100, N)

y, yerr = y_experiment(10.0, 3.0, sigma, x)

Let's look at our "experiment":

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=yerr, fmt="o")
plt.show()

Now we can write a function to do the fitting

In [ ]:
def linear_regression(x, y, yerr):
    """fit data (x_i, y_i) with errors {yerr_i} to a line"""
    
    N = len(x)

    C = np.sum(1.0 / yerr**2)

    S_x = np.sum(x / yerr**2)
    S_x2 = np.sum(x * x / yerr**2)

    S_y = np.sum(y / yerr**2)
    S_xy = np.sum(x * y / yerr**2)

    a2 = (C * S_xy - S_x * S_y)/(C * S_x2 - S_x**2)
    a1 = (S_y * S_x2 - S_xy * S_x) / (C * S_x2 - S_x**2)

    chisq = np.sum((a1 + a2 * x - y)**2 / yerr**2)
    chisq /= N-2

    return a1, a2, chisq

a1, a2, chisq = linear_regression(x, y, yerr)

Now we can look at how well our fit does:

In [ ]:
ax.plot(x, a1 + a2*x)
fig

## Pathologies

We need to be careful to not over-interpret a fit.  Consider the [Anscombe's quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet).  These 4 very different datasets all have the same linear fit (to a few digits significance).

# Testing Our Least Squares

We'll write a general linear least squares implementation and test it first on a parabolic
fit and then on a higher-order polynomial.

## Parabolic fit

Let's consider fitting to a function:

$$Y(x; \{a_j\}) = \sum_{j=1}^M a_j x^{j-1}$$

for $M = 3$, this is:

$$Y(x) = a_1 + a_2 x + a_3 x^2$$

If we have 10 data points, $x_1, \ldots, x_{10}$, then our matrix ${\bf A}$ will take the form:

$${\bf A} = \left (\begin{array}{ccc} 1/\sigma_1 & x_1/\sigma_1 & x_1^2/\sigma1 \\
                                      1/\sigma_2 & x_2/\sigma_2 & x_2^2/\sigma_2 \\
                                      \vdots & \vdots & \vdots \\
                                      1/\sigma_{10} & x_{10}/\sigma_{10} & x_{10}^2/\sigma_{10} 
                                      \end{array} \right )$$

Let's first write a function that takes an $x_i$ and returns the entries in a row of ${\bf A}$

In [ ]:
def basis(x, M=3):
    """ the basis function for the fit, x**n"""
    
    j = np.arange(M)
    return x**j

Now we'll write a function that takes our data and errors and sets up the linear system ${\bf A}^\intercal {\bf A} {\bf a} = {\bf A}^\intercal {\bf b}$ and solves it.

In [ ]:
def general_regression(x, y, yerr, M):
    """ here, M is the number of fitting parameters.  We will fit to
        a function that is linear in the a's, using the basis functions
        x**j """

    N = len(x)

    # construct the design matrix -- A_{ij} = Y_j(x_i)/sigma_i -- this is
    # N x M.  Each row corresponds to a single data point, x_i, y_i
    A = np.zeros((N, M), dtype=np.float64)

    for i in range(N):
        A[i,:] = basis(x[i], M) / yerr[i]

    # construct the MxM matrix for the linear system, A^T A:
    ATA = np.transpose(A) @ A

    print("condition number of A^T A:", np.linalg.cond(ATA))

    # construct the RHS
    b = np.transpose(A) @ (y / yerr)

    # solve the system
    a = np.linalg.solve(ATA, b)

    # return the chisq
    chisq = 0
    for i in range(N):
        chisq += (np.sum(a*basis(x[i], M)) - y[i])**2 / yerr[i]**2

    chisq /= N-M

    return a, chisq

Finally, we'll make up some experiment data that follows a parabola, but with a perturbation.

In [ ]:
def y_experiment2(a1, a2, a3, sigma, x):
    """ return the experimental data and error in a quadratic +
    random fashion, with a1, a2, a3 the coefficients of the
    quadratic and sigma is scale of the error.  This will be
    poorly matched to a linear fit for a3 != 0 """

    N = len(x)

    # this creates a Gaussian normal random number from a distribution
    # centered on 0 with a width of sigma
    rng = np.random.default_rng()
    r = sigma * rng.standard_normal(N)
    
    y = a1 + a2*x + a3*x*x + r

    return y, np.ones_like(y) * sigma

In [ ]:
N = 40
x = np.linspace(0, 100.0, N)

# one-sigma error
sigma = 8.0

y, yerr = y_experiment2(2.0, 1.50, -0.02, sigma, x)

In [ ]:
fig, ax = plt.subplots()

ax.errorbar(x, y, yerr=yerr, fmt="o")
fig

Now let's do the fit of a parabola.  Along the way, we'll print out the condition number of the matrix ${\bf A}^\intercal{\bf A}$.

In [ ]:
# do the regression with M = 3 (1, x, x^2)
M = 3
a, chisq = general_regression(x, y, yerr, M)

Notice that the condition number is quite large!

In [ ]:
ax.plot(x, a[0] + a[1]*x + a[2]*x*x)
fig

## Higher order fit

In [ ]:
fig, ax = plt.subplots()
M = 10
a, chisq = general_regression(x, y, yerr, M)
yfit = np.zeros((len(x)), dtype=x.dtype)
for i in range(N):
    base = basis(x[i], M)
    yfit[i] = np.sum(a*base)

ax.errorbar(x, y, yerr=yerr, fmt="o")
ax.plot(x, yfit)
fig

The $M = 10$ polynomial fits, but the condition number is large.  It is not clear that going higher order here is wise.

## Fitting and condition number

It can be shown that if your basis function is orthonormal in the interval you are fitting over, then the condition number of the matrix ${\bf A}^\intercal {\bf A}$ will be much lower.  For example, it we fit to the interval $[-1, 1]$, then the [Legendre polynomials](https://en.wikipedia.org/wiki/Legendre_polynomials) are a good basis.

The text by Yakowitz & Szidarovszky has a good discussion on this.

## Errors in the fitting parameters

It can also be shown that the errors in the fitting parameters are:

$$\sigma_{a_j} = \sqrt{C_{jj}}$$

where

$${\bf C} = ({\bf A}^\intercal {\bf A})^{-1}$$

is the _covariance matrix_. The covariance matrix is used often in fields like physical cosmology, to infer the parameters for the large-scale structure of the universe.

# Nonlinear Fitting

What about the case of fitting to a function where the fit parameters enter in a nonlinear fashion?
For example:

$$f(x; a_0, a_1) = a_0 e^{a_1 x}$$

Let's look at how we would fit directly to a nonlinear function.

We'll minimize the same fitting function:

$$\chi^2 = \sum_{i=1}^N \frac{(y_i - f(x; {\bf a}))^2}{\sigma_i^2}$$

with fitting parameters ${\bf a} = (a_1, \ldots, a_M)^\intercal$.

Now we take the derivatives with respect to each parameter, $a_k$:

$$\frac{\partial \chi^2}{\partial a_k} = -2 \sum_{i=1}^N \frac{(y_i - f(x, {\bf a}))}{\sigma_i^2} \frac{\partial f}{\partial a_k} = 0$$

Let's define $g_k \equiv {\partial \chi^2}/{\partial a_k}$, then we have

$${\bf g}({\bf a}) = \left ( \begin{array}{c} g_1({\bf a}) \\ g_2({\bf a}) \\ \vdots \\ g_M({\bf a}) \end{array} \right ) = 0$$

This is a nonlinear system of $M$ equations and $M$ unknowns.  We can solve this using the same multivariate Newton's method we looked at before:

* Take an initial guess at the fit parameters, ${\bf a}^{(k)}$
* Solve the system
  $${\bf J} \cdot \delta {\bf a} = -{\bf g}({\bf a}^{(k)}),$$
  where $J_{ij} = \partial g_i/\partial a_j$ is the Jacobian
* Correct the initial guess,
  $${\bf a}^{(k+1)} = {\bf a}^{(k)} + \delta {\bf a}$$

**Warning:** As we've seen with Newton's method, convergence will be very sensitive to the initial guess.

## Fitting an exponential

Let's try this out on data that is constructed to follow an exponential trend.

First let's construct the data, and perturb it with some errors.  We'll take the form:

$$y = a_0 e^{a_1 x}$$

In [ ]:
# make up some experimental data
a0 = 2.5
a1 = 2./3.
sigma = 4.0

N = 30
x = np.linspace(0.0, 5.0, N)

We will do a Gaussian-normal sampling of the error (with width $\sigma$), but in the fit, the uncertainty is just $\sigma$.

In [ ]:
rng = np.random.default_rng()
r = sigma * rng.standard_normal(N)
y = a0 * np.exp(a1 * x) + r
yerr = sigma * np.ones_like(r)

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=yerr, fmt="o")
fig

Now, let's compute our vector ${\bf g}$ that we will zero:

\begin{align*}
g_0 &= \frac{\partial \chi^2}{\partial a_0} = -2 \sum_{i=1}^N \frac{(y_i - a_0 e^{a_1 x_i})}{\sigma_i^2} (e^{a_1 x_i}) \\
g_1 &= \frac{\partial \chi^2}{\partial a_1} = -2 \sum_{i=1}^N \frac{(y_i - a_0 e^{a_1 x_i})}{\sigma_i^2} (x_i a_0 e^{a_1 x_i})
\end{align*}

We can divide out the $-2$ in each expression.  We'll keep the overall $a_0$ in the expression, to deal with the case where it might be $0$.  This gives:

\begin{align*}
g_0 &= \sum_{i=1}^N \frac{(y_i - a_0 e^{a_1 x_i})}{\sigma_i^2} (e^{a_1 x_i}) \\
g_1 &= a_0 \sum_{i=1}^N \frac{(y_i - a_0 e^{a_1 x_i})}{\sigma_i^2} (x_i e^{a_1 x_i})
\end{align*}

Let's write a function to compute this:

In [ ]:
def g(x, y, yerr, a):
    """compute the nonlinear functions we minimize.  Here a is the vector
    of fit parameters"""
    
    a0, a1 = a
    
    g0 = np.sum(np.exp(a1 * x) * (y - a0 * np.exp(a1 * x)) / yerr**2)
    g1 = a0 * np.sum(x * np.exp(a1 * x) * (y - a0 * np.exp(a1 * x)) / yerr**2)
    
    return np.array([g0, g1])

We also need the Jacobian.  We could either compute this numerically, via differencing, or analytically.  We'll do the latter.

\begin{align*}
\frac{\partial g_0}{\partial a_0} &= -\sum_{i=1}^N \frac{e^{2a_1 x_i}}{\sigma_i^2} \\
\frac{\partial g_0}{\partial a_1} &= \sum_{i=1}^N \frac{x_i e^{a_1 x_i} (y_i - 2 a_0 e^{a_1 x_i})}{\sigma_i^2} \\
\frac{\partial g_1}{\partial a_0} &=  \sum_{i=1}^N \frac{x_i e^{a_1 x_i} (y_i - 2 a_0 e^{a_1 x_i})}{\sigma_i^2} \\
\frac{\partial g_1}{\partial a_1} &= \sum_{i=1}^N \frac{a_0 x_i^2 e^{a_1 x_i} (y_i - 2 a_0 e^{a_1 x_i})}{\sigma_i^2}
\end{align*}

Notice that the Jacobian is symmetric:

$${\bf J} = \left ( \begin{array}{cc}
         \frac{\partial g_0}{\partial a_0} & \frac{\partial g_0}{\partial a_1} \\
         \frac{\partial g_1}{\partial a_0} & \frac{\partial g_1}{\partial a_1} \end{array} \right )
= \left ( \begin{array}{cc}
         \frac{\partial^2 \chi^2}{\partial a_0^2} & \frac{\partial^2 \chi^2}{\partial a_0 \partial a_1} \\
         \frac{\partial^2 \chi^2}{\partial a_1 \partial a_0} & \frac{\partial^2 \chi^2}{\partial a_1^2} \end{array} \right )       
$$

This is called the [Hessian matrix](https://en.wikipedia.org/wiki/Hessian_matrix).

Let's write this function:

In [ ]:
def jac(x, y, yerr, a):
    """ compute the Jacobian of the function g"""

    a0, a1 = a
    
    dg0da0 = -np.sum(np.exp(2.0 * a1 * x) / yerr**2)
    dg0da1 = np.sum(x * np.exp(a1 * x) * (y - 2.0 * a0 * np.exp(a1 * x)) / yerr**2)
    dg1da0 = dg0da1
    dg1da1 = np.sum(a0 * x**2 * np.exp(a1 * x) * (y - 2.0 * a0 * np.exp(a1 * x)) / yerr**2)
    
    return np.array([[dg0da0, dg0da1],
                     [dg1da0, dg1da1]])

**Tip:** Convergence here is very sensitive to the initial guess.  We'll do two modifications to deal with this:
* we add a cap on the number of iterations allowed prevent this from being an infinite loop if we can't converge
* we prevent our $[a_0, a_1]$ from changing by more than 20% per iteration

In [ ]:
def fit(aguess, x, y, yerr, tol=1.e-5):
    """ aguess is the initial guess to our fit parameters.  x and y
        are the vector of points that we are fitting to, and yerr are
        the errors in y"""
    
    avec = aguess.copy()

    max_steps = 100
    err = 1.e100
    n = 0
    while err > tol and n < max_steps:

        # get the jacobian
        J = jac(x, y, yerr, avec)

        # get the current function values
        gv = g(x, y, yerr, avec)

        # solve for the correction: J da = -g
        da = np.linalg.solve(J, -gv)

        # limit the change
        avec = np.clip(avec + da, 0.8 * avec, 1.2 * avec)
        err = np.max(np.abs(da/avec))
        n += 1
    return avec

In [ ]:
# initial guesses
aguess = np.array([2.0, 1.0])

# fit
afit = fit(aguess, x, y, yerr)
afit

In [ ]:
ax.plot(x, afit[0] * np.exp(afit[1] *x))
fig

## Is it a minimum?

We just found an extrema.  Let's plot the surface around our fit parameters to see if it looks like a minimum

In [ ]:
npts = 100
a0v = np.linspace(0.5 * afit[0], 2.0 * afit[0], npts)
a1v = np.linspace(0.5 * afit[1], 2.0 * afit[1], npts)

def chisq(a0, a1, x, y, yerr):
    return np.sum((y - a0 * np.exp(a1 * x))**2 / yerr**2)

chisq(afit[0], afit[1], x, y, yerr)

In [ ]:
c2 = np.zeros((npts, npts), dtype=np.float64)
for i, a0 in enumerate(a0v):
    for j, a1 in enumerate(a1v):
        c2[i, j] = chisq(a0, a1, x, y, yerr)

Now we'll plot the (log of) the $\chi^2$

In [ ]:
fig, ax = plt.subplots()

# we need to transpose to put a0 on the horizontal
# we use origin = lower to have the origin at the lower left
im = ax.imshow(np.log10(c2).T,
               origin="lower",
               extent=[a0v[0], a0v[-1], a1v[0], a1v[-1]])
fig.colorbar(im, ax=ax, orientation="horizontal")
ax.scatter([afit[0]], [afit[1]], color="r", marker="x")
ax.set_xlabel("$a_0$")
ax.set_ylabel("$a_1$")
fig

It looks like there is a very broad minimum there.

# Fitting Function in SciPy

In [ ]:
# make up some experimental data
a0 = 2.5
a1 = 2./3.
sigma = 4.0

N = 25

x = np.linspace(0.0, 4.0, N)

rng = np.random.default_rng()
r = sigma * rng.standard_normal(N)

y = a0 * np.exp(a1 * x) + r
yerr = sigma * np.ones_like(r)

fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=yerr, fmt="o")
fig

In [ ]:
def resid(avec, x, y, yerr):
    """ the residual function -- this is what will be minimized by the
        scipy.optimize.leastsq() routine.  avec is the parameters we
        are optimizing -- they are packed in here, so we unpack to
        begin.  (x, y) are the data points 

        scipy.optimize.leastsq() minimizes:

           x = arg min(sum(func(y)**2,axis=0))
                    y

        so this should just be the distance from a point to the curve,
        and it will square it and sum over the points
        """

    a0, a1 = avec

    # note: if we wanted to deal with error bars, we would weight each
    # residual accordingly
    return (y - a0 * np.exp(a1 * x)) / yerr

In [ ]:
# initial guesses
a0 = 0.5
a1 = 0.5

# fit -- here the args is a tuple of objects that will be added to the
# argument lists for the function to be minimized (resid in our case)
afit, flag = optimize.leastsq(resid, [a0, a1], args=(x, y, yerr))

print(flag)
print(afit)

In [ ]:
ax.plot(x, afit[0] * np.exp(afit[1] * x))
fig

# Exercises: Estimating $H_0$ from Type Ia Supernovae

Type Ia supernova are used as standardizable candles to measure cosmological distances.  By observing the lightcurve and measuring how long it takes for the supernova to dim we can empirically determine its brightness via the [Phillips relation](https://en.wikipedia.org/wiki/Phillips_relationship).

The paper [_Spectra and Hubble Space Telescope Light Curves of Six Type Ia Supernovae at 0.511 < z < 1.12 and the Union2 Compilation_](https://ui.adsabs.harvard.edu/abs/2010ApJ...716..712A/abstract) by Amanullah et al. made a [data set available](https://supernova.lbl.gov/Union/figures/SCPUnion2_mu_vs_z.txt) that has ~ 500 Type Ia supernovae.

That paper does a far, far more sophisticated analysis than we do, and they fit for other cosmological parameters that we will, so we will not get the same answer as they do.  But this is a good dataset to try out regression.

**Note:** if you are lost, some very good hints on how to complete these exercises are available here\
https://zingale.github.io/computational_astrophysics/fitting/application-snia-h0.html

## Exercise 1: Load dataset (2 pt)

The dataset has 4 columns:
* supernova identifier
* redshift, $z$, due to cosmological expansion
* [distance modulus](https://en.wikipedia.org/wiki/Distance_modulus), $\mu$, defined as:

  $$\mu = m - M = 5 \log_{10} \left (\frac{d}{10~\mbox{pc}}\right )$$

  where $m$ is the [apparent magnitude](https://en.wikipedia.org/wiki/Apparent_magnitude) of the
  supernova (what we observe) and $M$ is the [absolute magnitiude](https://en.wikipedia.org/wiki/Absolute_magnitude) of the supernova (inferred empirically from the lightcurve).
* uncertainty in $\mu$

Download the dataset by making sure that you are in the `lectures` directory, and then typing into your terminal the command
```
curl -O https://supernova.lbl.gov/Union/figures/SCPUnion2_mu_vs_z.txt
```

Load the dataset into python with the command
```python
data = np.genfromtxt("SCPUnion2_mu_vs_z.txt",
                     dtype=[("name", "S6"), ("z", "f8"), ("mu", "f8"), ("dmu", "f8")])
```
and sort the data based on $z$ with
```python
idx_sort = np.argsort(data["z"])
zs = data["z"][idx_sort]
mus = data["mu"][idx_sort]
dmus = data["dmu"][idx_sort]
```
plot the distance modulus with redshift, using the commands:
```python
fig, ax = plt.subplots()
ax.errorbar(data["z"], data["mu"], yerr=data["dmu"], fmt="o", ms=4)
ax.set_xlabel("z")
ax.set_ylabel(r"$\mu$")
ax.grid()
fig
```

In [ ]:
## Answer here


## Exercise 2 (2 pt)

Make a plot that looks a bit more like a [Hubble diagram](https://en.wikipedia.org/wiki/Hubble%27s_law#Hubble_diagram) by plotting redshift vs. distance,
computing distance as:

$$d = 10 \cdot 10^{\mu/5}~\mathrm{pc}$$

Display the result below, using the same code as above

## Exercise 3 (2 pt) - Clip to low-redshift data

We have the [distance modulus](https://en.wikipedia.org/wiki/Distance_modulus), which is related to the magnitudes via:

$$\mu = m - M = 5 \log_{10} \left (\frac{d}{10~\mbox{pc}}\right )$$

Since cosmologists usually work in terms of Mpc, let's rewrite this as:

$$\mu = m - M = 5 \log_{10} \left (\frac{d}{10~\mbox{pc}}\right ) 
    + 5 \log_{10} \left ( \frac{1~\mbox{Mpc}}{1~\mbox{Mpc}}\right )
    = 5 \log_{10} \left (\frac{d}{1~\mbox{Mpc}}\right ) + 25$$

Now, in an expanding Universe, the distance that goes here is the [luminosity distance](https://en.wikipedia.org/wiki/Luminosity_distance) which can be expressed via an expansion in redshift as (for $z \ll 1$):

$$d_L \approx \frac{c}{H_0} \left [ z + \frac{1}{2} (1 - q_0) z^2 \right]$$

Here $H_0$ is the [Hubble constant](https://en.wikipedia.org/wiki/Hubble%27s_law) and $q_0$ is the [deceleration parameter](https://en.wikipedia.org/wiki/Deceleration_parameter). We will use this to fit for small $z$, to estimate $H_0$.

First we need to select only the _low redshift_ data, where our expansion of distance luminosity might apply.  We'll consider $z < 0.2$.

Clip to low redshifts using the commands
```python
zmax = 0.2
idx = zs < zmax
z_low = zs[idx]
mu_low = mus[idx]
dmu_low = dmus[idx]
```
and plot $z$ versus $\mu$ for $z<0.2$, the same way as before.

In [ ]:
## Answer here


### Exercise 4 (2 pt) - Write the residual function

We'll use the SciPy fitting routine to estimate $H_0$ from this data. We want to fit:

$$\mu = 5\log_{10} \left (\frac{cz}{H_0 \cdot 1~\mbox{Mpc}} \left [1 + \frac{1}{2} (1 - q_0) z \right ] \right ) + 25$$

which we'll write as:

$$\mu = 5\log_{10} \left (a_0 z \left [1 + \frac{1}{2} (1 - a_1) z \right ] \right ) + 25 \qquad{(1)}
$$

This is a nonlinear expression in terms of the fit parameters, $a_0$, $a_1$.  

Write the residual function `resid(avec, z, mu, dmu)`, with inputs `avec = [avec[0], avec[1]]` for the two fitting parameters, as well as inputs `z`, `mu`, and `dmu` for the redshift, magnitude, and magnitude error.

e.g. this is the "$\chi$" in your "$\chi^2$" fit for the function in equation (1) above.

In [ ]:
## Answer here


### Exercise 5 (2 pt) - Fit the data, and recover Hubble's constant

Once we get $a_0$, we can get Hubble's constant as:

$$H_0 = \frac{c}{a_0 \cdot 1~\mbox{Mpc}}$$

Now, take initial guesses for the fit to be
```python
# imagine H0 = 50
c = 3.e5   # km/s 
H0_guess = 50   # km/s/Mpc

a0 = c / H0_guess
a1 = 1
```

use `optimize.leastsq` to fit your function to the redshift and magnitude data.

In [ ]:
# Answer here
